# Paper 2 — Field Site Augmentation for RF v3 (32 spatial points)

**SCRIPT: Paper2_RF_v3_Field_Sites_Augmentation.ipynb**

**Purpose:** the RF model currently trains on only **2 ground-truth locations** (stations 24T,
25T), which is why spatial-block cross-validation collapsed to near-zero R² (0.039) —
with only 2 points, spatial CV is essentially leave-one-station-out, giving the model no
real spatial pattern to learn from.

This notebook extracts the same covariate set (AOD, LST, meteorology, fire, distance-to-road,
distance-to-facility, land-use) for the **32 field-campaign sites** from Paper 1
(`field_sites_32_merged.csv`, June 2023, each site measured once), producing a
32-row supplement with the exact same schema as the main RF training table. Concatenating
this onto the 7,304-row station dataset gives the model **34 spatially distinct locations**
instead of 2, making spatial-block CV meaningful for the first time.

**Run this AFTER** `Paper2_RF_v2_Additional_Covariates.ipynb` has been run at least through
Step 5 in the same Colab session (so Earth Engine is already authenticated, `roads_gdf` and
`facilities` are already loaded) — this notebook reuses those rather than rebuilding them.
If running standalone instead, see the TODOs below to re-authenticate/re-load.

**Output:** `field_sites_32_with_covariates.csv` — concatenate this onto the RF v2 `df`
before Step 6 (feature prep / dropna) and re-run model training.


In [ ]:
# SECTION: Setup & defensive imports
import sys, subprocess

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg, _imp in [("geopandas", "geopandas"), ("requests", "requests"), ("osmnx", "osmnx")]:
    _ensure(_pkg, _imp)

import pandas as pd
import numpy as np
import requests
import time as _time
from datetime import datetime, timedelta


## Step 0 — Earth Engine auth (skip if already authenticated in this session)


In [ ]:
# SECTION: Earth Engine auth (defensive -- reuses existing session if already authenticated)
import ee

EE_PROJECT_ID = "saraburi-thesis"  # confirmed Cloud Project ID

try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
print("Earth Engine ready, project:", EE_PROJECT_ID)


## Step 1 — Load the 32 field sites


In [ ]:
# SECTION: Load field site data
from google.colab import files
import os

if os.path.exists("field_sites_32_merged.csv"):
    field = pd.read_csv("field_sites_32_merged.csv")
else:
    print("Upload field_sites_32_merged.csv:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    field = pd.read_csv(fname)

field["date"] = pd.to_datetime(field["date"])
print(field.shape)
field[["site_id", "lat", "lon", "date", "pm25_field"]]


## Step 2 — AOD (MODIS MAIAC) at each site-date

Only 32 point-date combinations total, so no chunking needed here (well under the
5,000-element GEE limit). Uses a ±3-day window around the field-measurement date since
same-day AOD is frequently missing due to cloud cover (matches the tolerance already
documented as a limitation for the main dataset).


In [ ]:
# SECTION: AOD at each field site (±3-day window per site-date)
def get_aod_point(lat, lon, date, window_days=3):
    point = ee.Geometry.Point([lon, lat])
    d = ee.Date(date.strftime("%Y-%m-%d"))
    coll = (ee.ImageCollection("MODIS/061/MCD19A2_GRANULES")
            .filterDate(d.advance(-window_days, "day"), d.advance(window_days + 1, "day"))
            .filterBounds(point)
            .select(["Optical_Depth_047", "Optical_Depth_055"]))
    img = coll.mean()
    val = img.reduceRegion(ee.Reducer.mean(), point, 1000).getInfo()
    aod_047 = val.get("Optical_Depth_047")
    aod_055 = val.get("Optical_Depth_055")
    return (
        aod_047 * 0.001 if aod_047 is not None else np.nan,
        aod_055 * 0.001 if aod_055 is not None else np.nan,
    )

aod_047_list, aod_055_list = [], []
for _, row in field.iterrows():
    print(f"AOD for {row.site_id} ({row.date.date()})...")
    for attempt in range(3):
        try:
            a047, a055 = get_aod_point(row.lat, row.lon, row.date)
            break
        except Exception as e:
            print(f"  retry {attempt+1}/3: {e}")
            _time.sleep(5)
            a047, a055 = np.nan, np.nan
    aod_047_list.append(a047)
    aod_055_list.append(a055)
    _time.sleep(0.3)

field["aod_047"] = aod_047_list
field["aod_055"] = aod_055_list
print("AOD missing:", field["aod_047"].isna().sum(), "/", len(field))


## Step 3 — LST (MODIS MOD11A2) at each site-date


In [ ]:
# SECTION: LST at each field site (nearest 8-day composite)
def get_lst_point(lat, lon, date, window_days=8):
    point = ee.Geometry.Point([lon, lat])
    d = ee.Date(date.strftime("%Y-%m-%d"))
    coll = (ee.ImageCollection("MODIS/061/MOD11A2")
            .filterDate(d.advance(-window_days, "day"), d.advance(window_days + 1, "day"))
            .filterBounds(point)
            .select("LST_Day_1km"))
    img = coll.mean()
    val = img.reduceRegion(ee.Reducer.mean(), point, 1000).getInfo().get("LST_Day_1km")
    return val * 0.02 - 273.15 if val is not None else np.nan

lst_list = []
for _, row in field.iterrows():
    print(f"LST for {row.site_id}...")
    for attempt in range(3):
        try:
            lst_list.append(get_lst_point(row.lat, row.lon, row.date))
            break
        except Exception as e:
            print(f"  retry {attempt+1}/3: {e}")
            _time.sleep(5)
    else:
        lst_list.append(np.nan)
    _time.sleep(0.3)

field["lst_c"] = lst_list
print("LST missing:", field["lst_c"].isna().sum(), "/", len(field))


## Step 4 — NDVI / EVI (MODIS MOD13Q1) at each site-date


In [ ]:
# SECTION: NDVI/EVI at each field site (nearest 16-day composite)
def get_vi_point(lat, lon, date, window_days=16):
    point = ee.Geometry.Point([lon, lat])
    d = ee.Date(date.strftime("%Y-%m-%d"))
    coll = (ee.ImageCollection("MODIS/061/MOD13Q1")
            .filterDate(d.advance(-window_days, "day"), d.advance(window_days + 1, "day"))
            .filterBounds(point)
            .select(["NDVI", "EVI"]))
    img = coll.mean()
    val = img.reduceRegion(ee.Reducer.mean(), point, 250).getInfo()
    ndvi = val.get("NDVI")
    evi = val.get("EVI")
    return (
        ndvi * 0.0001 if ndvi is not None else np.nan,
        evi * 0.0001 if evi is not None else np.nan,
    )

ndvi_list, evi_list = [], []
for _, row in field.iterrows():
    print(f"NDVI/EVI for {row.site_id}...")
    for attempt in range(3):
        try:
            n, e = get_vi_point(row.lat, row.lon, row.date)
            break
        except Exception as ex:
            print(f"  retry {attempt+1}/3: {ex}")
            _time.sleep(5)
            n, e = np.nan, np.nan
    ndvi_list.append(n)
    evi_list.append(e)
    _time.sleep(0.3)

field["ndvi"] = ndvi_list
field["evi"] = evi_list


## Step 5 — Land-use (ESA WorldCover) at each site


In [ ]:
# SECTION: Built-up fraction at each field site (static, doesn't vary by date)
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
builtup_mask = worldcover.eq(50)

builtup_list = []
for _, row in field.iterrows():
    point = ee.Geometry.Point([row.lon, row.lat]).buffer(500)
    val = builtup_mask.reduceRegion(ee.Reducer.mean(), point, 10).getInfo().get("Map")
    builtup_list.append(val)
    _time.sleep(0.2)

field["builtup_frac"] = builtup_list
print(field[["site_id", "builtup_frac"]])


## Step 6 — Meteorology (NASA POWER) at each site-date


In [ ]:
# SECTION: Meteorology at each field site (single-day point query, no API key needed)
POWER_PARAMS = ["T2M", "WS10M", "WS50M", "WD10M", "RH2M", "PRECTOTCORR", "PS"]

def get_power_point(lat, lon, date):
    date_str = date.strftime("%Y%m%d")
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": ",".join(POWER_PARAMS), "community": "AG",
        "longitude": lon, "latitude": lat,
        "start": date_str, "end": date_str, "format": "JSON",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()["properties"]["parameter"]
    return {k: list(v.values())[0] for k, v in data.items()}

met_rows = []
for _, row in field.iterrows():
    print(f"Meteorology for {row.site_id}...")
    try:
        met_rows.append(get_power_point(row.lat, row.lon, row.date))
    except Exception as e:
        print(f"  failed: {e}")
        met_rows.append({p: np.nan for p in POWER_PARAMS})
    _time.sleep(0.5)

met_df = pd.DataFrame(met_rows).rename(columns={
    "T2M": "temperature", "WS10M": "wind_speed_10m", "WS50M": "wind_speed_50m",
    "WD10M": "wind_direction", "RH2M": "relative_humidity",
    "PRECTOTCORR": "rainfall", "PS": "surface_pressure",
})
met_df = met_df.replace(-999, np.nan)
field = pd.concat([field.reset_index(drop=True), met_df.reset_index(drop=True)], axis=1)


## Step 7 — Fire count + distance to nearest fire (NASA FIRMS)

Reuses the same `FIRMS_MAP_KEY` from the rebuild notebook. Each field site only needs a
single ±3-day window around its measurement date, so this is 32 small requests total
(well under any rate limit).


In [ ]:
# SECTION: Fire count + nearest-fire distance at each field site
FIRMS_MAP_KEY = "fd097afb1b95210ea55e6d38be633ab9"  # TODO -- same key used before

from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

def get_fire_near_point(lat, lon, date, window_days=3, buffer_deg=0.5, map_key=FIRMS_MAP_KEY):
    bbox = f"{lon-buffer_deg},{lat-buffer_deg},{lon+buffer_deg},{lat+buffer_deg}"
    start = (date - timedelta(days=window_days)).strftime("%Y-%m-%d")
    span = window_days * 2 + 1  # this MAP_KEY's tier caps at 5 -- window_days<=2 keeps span<=5
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/MODIS_SP/{bbox}/{min(span,5)}/{start}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        from io import StringIO
        df = pd.read_csv(StringIO(r.text))
    except Exception as e:
        print(f"  fire fetch failed: {e}")
        return 0, np.nan
    if df.empty:
        return 0, np.nan
    dists = df.apply(lambda r: haversine_km(lat, lon, r["latitude"], r["longitude"]), axis=1)
    return len(df), dists.min()

fire_counts, fire_dists = [], []
for _, row in field.iterrows():
    print(f"Fire data for {row.site_id}...")
    fc, fd = get_fire_near_point(row.lat, row.lon, row.date, window_days=2)  # span=5, matches MAP_KEY limit
    fire_counts.append(fc)
    fire_dists.append(fd)
    _time.sleep(1)

field["fire_count"] = fire_counts
field["dist_to_nearest_fire_km"] = fire_dists
# Same interpretation as the main dataset: no detection in window -> treat as 0 fires,
# distance imputed with the farthest observed distance (see RF v2 Step 6 for the same logic)


## Step 8 — Distance to nearest major road (reuses `roads_gdf` from RF v2 if present)


In [ ]:
# SECTION: Distance to nearest major road
import geopandas as gpd
from shapely.geometry import Point

if "roads_gdf" not in dir():
    print("roads_gdf not found in session -- rebuilding (TODO: confirm bbox matches RF v2 Step 3)")
    import osmnx as ox
    north, south, east, west = 14.95, 14.25, 101.45, 100.55
    _bbox = (west, south, east, north)
    roads_gdf = ox.graph_to_gdfs(
        ox.graph_from_bbox(_bbox, network_type="drive",
                            custom_filter='["highway"~"motorway|trunk|primary|secondary"]'),
        nodes=False, edges=True
    )

points_gdf = gpd.GeoDataFrame(
    field, geometry=gpd.points_from_xy(field.lon, field.lat), crs="EPSG:4326"
).to_crs("EPSG:32647")
roads_utm = roads_gdf.to_crs("EPSG:32647")

field["dist_to_road_km"] = points_gdf.geometry.apply(lambda p: roads_utm.distance(p).min() / 1000)
print(field[["site_id", "dist_to_road_km"]])


## Step 9 — Distance to nearest DIW facility (reuses `facilities` from RF v2 if present)


In [ ]:
# SECTION: Distance to nearest geocoded facility
if "facilities" not in dir():
    print("facilities not found in session -- upload diw_facilities_geocoded.csv")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    facilities = pd.read_csv(fname)

fac_gdf = gpd.GeoDataFrame(
    facilities, geometry=gpd.points_from_xy(facilities.lon, facilities.lat), crs="EPSG:4326"
).to_crs("EPSG:32647")

_dist_to_facility = points_gdf.geometry.apply(lambda p: fac_gdf.distance(p).min() / 1000)
# All 295 facilities loaded here are already the "high-risk" subset (Cement/Quarry/Stone +
# Power/Energy + Waste/Recycling). The main RF pipeline ended up with THREE differently-named
# columns that are all computed from this same file at different points in its history --
# assign the same value to all three so the schema matches exactly (harmless redundancy,
# not a bug: dist_to_nearest_factory_km and dist_to_risk_factory_km come from the original
# v1 rebuild notebook; dist_to_nearest_facility_km is RF v2's own separately-added version).
field["dist_to_nearest_factory_km"] = _dist_to_facility
field["dist_to_risk_factory_km"] = _dist_to_facility
field["dist_to_nearest_facility_km"] = _dist_to_facility


## Step 10 — Finalize schema to match the main RF training table exactly


In [ ]:
# SECTION: Assemble final columns matching the main table's schema
field["month"] = field["date"].dt.month
field["day_of_week"] = field["date"].dt.dayofweek
field["station"] = field["site_id"]  # reuse this column name as the location identifier
field["pm25"] = field["pm25_field"]
field["pm10"] = field["pm10_field"]

final_cols = [
    "date", "station", "pm25", "pm10", "aod_047", "aod_055", "evi", "ndvi",
    "temperature", "wind_speed_10m", "wind_speed_50m", "wind_direction",
    "relative_humidity", "rainfall", "surface_pressure", "fire_count",
    "dist_to_nearest_fire_km", "month", "day_of_week",
    "dist_to_nearest_factory_km", "dist_to_risk_factory_km", "lat", "lon",
    "lst_c", "dist_to_road_km", "dist_to_nearest_facility_km", "builtup_frac",
]
field_final = field[final_cols].copy()
field_final["date"] = field_final["date"].dt.strftime("%Y-%m-%d")

print(field_final.shape)
print("Missing values per column:")
print(field_final.isna().mean().round(3))
field_final.to_csv("field_sites_32_with_covariates.csv", index=False)

try:
    files.download("field_sites_32_with_covariates.csv")
except Exception as e:
    print("Download skipped (not in Colab):", e)

print()
print("Done. In the RF v2 notebook, right before Step 6 (feature prep), add:")
print('  field_supp = pd.read_csv("field_sites_32_with_covariates.csv")')
print('  df = pd.concat([df, field_supp], ignore_index=True)')
print("then re-run Step 6 onward. This gives spatial-block CV 34 distinct locations")
print("instead of 2, making it a meaningful test for the first time.")
